## Recap

First, we split the data into training and test.

In [ ]:
import pandas as pd

df = pd.read_csv("https://dlsun.github.io/pods/data/bordeaux.csv",
                 index_col="year")
df_train = df.loc[:1980].copy()
df_test = df.loc[1981:].copy()

In [ ]:
X_train = df_train[["win", "summer"]]
y_train = df_train["price"]

X_test = df_test[["win", "summer"]]

Next, we fit a 5-nearest neighbors model to the training data and use the model to predict the labels on the test data.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

pipeline = make_pipeline(
          StandardScaler(),
          KNeighborsRegressor(n_neighbors=5))
pipeline.fit(X=X_train, y=y_train)
pipeline.predict(X=X_test)

## Training Error

On the training data, we know the true labels $y_1, y_2, ..., y_n$, so we can calculate the training error of our predictions ${\hat y}_1, {\hat y}_2, ..., {\hat y}_n$:

$$ \text{training MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat y_i)^2 $$

In [ ]:
pipeline.fit(X_train, y_train)
y_train_ = pipeline.predict(X_train)
((y_train - y_train_) ** 2).mean()

There's also a Scikit-Learn function for that!

In [ ]:
from sklearn.metrics import mean_squared_error
mean_squared_error(y_train, y_train_)

To see the problem with training error, let's calculate the training error of a $1$-nearest neighbor model.

In [ ]:
pipeline = make_pipeline(
          StandardScaler(),
          KNeighborsRegressor(n_neighbors=1))
pipeline.fit(X=X_train, y=y_train)
y_train_ = pipeline.predict(X=X_train)
mean_squared_error(y_train, y_train_)

A $1$-nearest neighbors model will always be perfect on the training data!

## Estimating Test Error

First, we choose half of the vintages in the training data to be in the training set. The remaining vintages will be in the validation set.

We can set a [random seed](https://en.wikipedia.org/wiki/Random_seed) to ensure reproducible results.



In [ ]:
import numpy as np

np.random.seed(0)
inds_train_set = np.random.choice(df_train.index, size=len(df_train) // 2,
                                  replace=False)
inds_train_set

In [ ]:
df_train_set = df_train.loc[inds_train_set]
df_val_set = df_train.drop(inds_train_set, axis="rows")

Notice that since both the training and validation sets came from the training data, both the inputs $X$ and the labels $y$ are known.

In [ ]:
X_train_set, y_train_set = df_train_set[["win", "summer"]], df_train_set["price"]
X_val_set, y_val_set = df_val_set[["win", "summer"]], df_val_set["price"]

Now, let's fit a 1-nearest neighbors model to the training set and calculate the MSE of the predictions on the validation set.

In [ ]:
pipeline = make_pipeline(
          StandardScaler(),
          KNeighborsRegressor(n_neighbors=1))
pipeline.fit(X_train_set, y_train_set)
y_val_set_ = pipeline.predict(X_val_set)
mean_squared_error(y_val_set, y_val_set_)

Notice that the validation MSE is no longer 0!

## Cross-Validation

Previously, we fit the model to the training set and evaluated the predictions on the validation set.

In [ ]:
pipeline = make_pipeline(
          StandardScaler(),
          KNeighborsRegressor(n_neighbors=1))
pipeline.fit(X_train_set, y_train_set)
y_val_set_ = pipeline.predict(X_val_set)
mean_squared_error(y_val_set, y_val_set_)

Now let's do the same thing, with the roles of the training and validation sets reversed.

In [ ]:
pipeline = make_pipeline(
          StandardScaler(),
          KNeighborsRegressor(n_neighbors=1))
pipeline.fit(X_val_set, y_val_set)
y_train_set_ = pipeline.predict(X_train_set)
mean_squared_error(y_train_set, y_train_set_)

We get two very different answers! To come up with one overall estimate of test error, we can average them.

In [ ]:
(195.71 + 306.92) / 2

We can let Scikit-Learn handle cross-validation, using the `cross_val_score` function.

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    pipeline,
    X=df_train[["win", "summer"]],
    y=df_train["price"],         # this is all of the training data!
    scoring="neg_mean_squared_error", # higher is better for a score
    cv=4)
scores

In [ ]:
# To get the MSE, we need to negate the negative MSE!
-scores.mean()